<a href="https://colab.research.google.com/github/agneym/herdr-needle-research/blob/main/colab_herdr_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Needle 2 as a Herdr expert (GPU)

This notebook trains the LoRA adapter that makes Needle 2 a Herdr terminal-multiplexer expert (25 Herdr operations: pane split/run/read/wait, agent start/get/read/wait, workspace/tab/session/worktree, integration, status).

1. Runtime > Change runtime type > **GPU** (T4 is fine).
2. Run the cells in order.
3. Upload `data.jsonl` when the canvas folder opens.
4. Download `adapter.pkl` at the end, then build locally:
   `needle build checkpoints/needle2.pkl --lora adapter.pkl --out tuned.cact`
   and query it with `python ask_herdr.py --weights tuned.cact --query "..."`.


In [1]:
!pip install -q "cactus-needle[gpu]"
import jax
print('jax backend:', jax.default_backend())
assert jax.default_backend() == 'gpu', 'Runtime is not on a GPU - re-select Runtime > Change runtime type > GPU'
print('GPU ready')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.3 MB/s eta 0:00:00
jax backend: gpu
GPU ready


## 1. Upload the dataset
Click **Choose file** and upload the `data.jsonl` from the project folder.


In [2]:
from google.colab import files
print('Upload data.jsonl:')
files.upload()
import os
assert os.path.exists('data.jsonl'), 'data.jsonl not found after upload'
print('lines:', sum(1 for _ in open('data.jsonl')))


Upload data.jsonl:


Saving data.jsonl to data.jsonl
lines: 225


## 2. Train the LoRA adapter
Full 25-tool catalogue, seq_len 4096, batch 2 (safe on a 16GB T4). If it OOMs it retries with batch 1.

In [ ]:
import shutil, subprocess

def make_cmd(batch):
    return ["needle",
        "finetune", "data.jsonl",
        "--epochs", "12",
        "--batch-size", str(batch),
        "--max-len", "4096",
        "--out", "adapter.pkl",
        "--lora-rank", "16",
        "--lora-alpha", "32",
        "--lr", "1e-4",
        "--val-split", "0.1",
    ]

assert shutil.which('needle'), 'needle CLI not on PATH'
for batch in (2, 1):
    print('running batch-size', batch)
    r = subprocess.run(make_cmd(batch))
    if r.returncode == 0:
        print('TRAINING OK')
        break
    print('batch', batch, 'failed (exit', r.returncode, ') - retrying smaller')
else:
    raise SystemExit('training failed at batch 1 as well')

import os
assert os.path.exists('adapter.pkl'), 'adapter.pkl not produced'
print('adapter.pkl size:', os.path.getsize('adapter.pkl'), 'bytes')


running batch-size 2


In [ ]:
print('Downloading adapter.pkl to your computer...')
from google.colab import files
files.download('adapter.pkl')
print('Done. On the target machine run:')
print('  needle build checkpoints/needle2.pkl --lora adapter.pkl --out tuned.cact')
print('  python ask_herdr.py --weights tuned.cact --query "split my pane to the right"')
